#Sistema di raccomandazione
Il sistema implementa un approccio di raccomandazione one-time, evitando la costruzione di matrici di similarità NxN. Applica un filtro hard sulla categoria e combina una similarità basata su ingredienti con una funzione deterministica di brand e segmento, pesate rispettivamente 0.8 e 0.2.



In [7]:
import pandas as pd
import numpy as np



1) Importo il df e creo un nuovo df con le colonne d'interesse



In [9]:
df = pd.read_csv("brand_categoria.csv",dtype={"code": str})

df = df[['code', 'product_name','brand_name', 'brand_segment','macro_category']].copy()
df['code'] = df['code'].astype(str)
df['product_name'] = df['product_name'].astype(str)
df['brand_name'] = df['brand_name'].astype(str)
df['brand_segment'] = df['brand_segment'].astype(str)
df['macro_category'] = df['macro_category'].astype(str)


df.tail(30)



FileNotFoundError: [Errno 2] No such file or directory: 'brand_categoria.csv'

In [ ]:
df_ing = pd.read_csv("ingr_cat.csv",dtype={"code": str})


df_ing['code'] = df_ing['code'].astype(str)
df_ing['product_name'] = df_ing['product_name'].astype(str)
df_ing['macro_category'] = df_ing['macro_category'].astype(str)
df_ing['inci_name'] = df_ing['inci_name'].astype(str)

df_finale = (
    df_ing
    .groupby(['code', 'product_name', 'macro_category'])['inci_name']
    .apply(lambda lst: ", ".join(sorted(set(lst))))
    .reset_index()
    .rename(columns={'inci_name': 'ingredients'}))

df_finale.tail(20)


,code,product_name,macro_category,ingredients
15111,96121214,Powder soft anti perspirant,Other,"Alpha-Isomethyl Lonone, Aluminum Zirconium Tet..."
15112,96125694,REXONA Déodorant Homme Bille Anti Transpirant ...,Deodorants,"Alpha-Isomethyl Ionone, Aluminum Zirconium Tet..."
15113,96125748,Motionsense active shield,Other,"Alpha-Lsomethyl Lonone, Aluminum Chlorohydrate..."
15114,96130339,Dove zero original 50ml,Body_Care,"Alpha-Isomethyl Ionone, Benzyl Alcohol, Citric..."
15115,96132289,Axe Déodorant Anti Bactérien You Spray 100ml,Deodorants,"Alcohol Denat, Alpha-Isomethyl Ionone, Butane,..."
15116,96132876,Monsavon Déodorant Femme Spray Compressé Fleur...,Hygiene,"Alpha-Isomethyl Ionone, Benzyl Alcohol, Bht, B..."
15117,96132906,Monsavon Déodorant Femme Spray Compressé Grena...,Hygiene,"Alpha-Isomethyl Ionone, Benzyl Alcohol, Bht, B..."
15118,96134696,DOVE Déodorant Femme Spray Compressé Pierre d'...,Body_Care,"Alpha-Isomethyl Ionone, Benzyl Alcohol, Benzyl..."
15119,96148402,Monsavon Déodorant Femme Spray Compressé Vanil...,Hygiene,"Benzyl Alcohol, Bht, Butane, Butyrospermum Par..."
15120,96183281,sensodyne,Other,"Aroma, Carbomer, Cocamidopropyl Betaine, Die 7..."


In [ ]:
#per la semplificazione della ricerca in fase di test, è una lista di "code" in comune tra i dataset
common_codes = set(df['code']).intersection(set(df_finale['code']))

print(f"Prodotti in df: {len(df)}")
print(f"Prodotti in df_finale: {len(df_finale)}")
print(f"Prodotti comuni: {len(common_codes)}")
common_codes

Prodotti in df: 31056
Prodotti in df_finale: 15131
Prodotti comuni: 12304


{'8001090102997',
 '3574661261065',
 '4005808246038',
 '3600542075091',
 '3560071132774',
 '8432225096735',
 '3257982124980',
 '4003573020051',
 '3600541291447',
 '3600522401032',
 '4058172938740',
 '6002413076119',
 '5028197823160',
 '3450970117739',
 '3153716003432',
 '0663350065558',
 '3600523399826',
 '8431876282061',
 '3600523583270',
 '8480000464170',
 '3250390448730',
 '8001090365255',
 '8717163651735',
 '8720181001857',
 '4010355505347',
 '8480000468161',
 '0878941001741',
 '4005808283804',
 '8717163732465',
 '8710908571596',
 '3614229376777',
 '4335619174795',
 '5420008514814',
 '3245678043844',
 '3560070337651',
 '3511720128303',
 '3600522448471',
 '4010355340450',
 '4006000132303',
 '9312657010013',
 '0634084471700',
 '2017017699996',
 '0697029421437',
 '8993137723756',
 '3478820044035',
 '3600541540262',
 '8412300836586',
 '8720181092763',
 '4021457606672',
 '0850005911395',
 '2000000002886',
 '6111273490154',
 '7640183490422',
 '3153716007966',
 '3662217011759',
 '33378755

**2. Assegnazione pesi ai brand_segment.**

Il sistema di suggerimento tiene conto del segmento di mercato e pesa la similarità in base ad esso.
Prodotti dello stesso segmento avranno peso maggiore.

In [ ]:
SEGMENT_SIMILARITY = {
    ('middle', 'middle'): 0.7,
    ('mass_market', 'mass_market'): 0.7,
    ('luxury', 'luxury'): 0.7,

    ('mass_market', 'middle'): 0.5,
    ('middle', 'mass_market'): 0.5,

    ('mass_market', 'luxury'): 0.3,
    ('luxury', 'mass_market'): 0.3,

    ('middle', 'luxury'): 0.5,
    ('luxury', 'middle'): 0.5
}


**3. Creazione dell'array di similarità 1xK.**

Per il brand non uso una cosine similarity, ma una funzione deterministica basata su regole di business che tengono conto sia dell’identità del brand sia del segmento di mercato.

Il sistema applica un hard filter sulla categoria per ridurre lo spazio di ricerca e garantire coerenza semantica, e successivamente calcola una similarità pesata basata su ingredienti e su regole di brand e segmento.



In [ ]:
def brand_segment_similarity_1xK(
    query_product_id,
    df,
    category_col='macro_category',
    segment_col='brand_segment'
):
    query_product_id = str(query_product_id)

    if query_product_id not in df['code'].values:
        raise ValueError("Codice prodotto non trovato")

    query_row = df[df['code'] == query_product_id].iloc[0]
    query_category = query_row[category_col]
    query_brand = query_row['brand_name']
    query_segment = query_row[segment_col]
    query_product_name = query_row['product_name']  # ← AGGIUNTA

    # Filtro hard: stessa categoria -> ricerca tra prodotti della stessa categoria
    df_filtered = df[df[category_col] == query_category].copy()
    df_filtered = df_filtered[df_filtered['code'] != query_product_id]
    #filtro per evitare che restituisca lo stesso prodotto
    df_filtered['query_brand_name'] = query_brand
    df_filtered['query_product_name'] = query_product_name

    similarities = []

    for _, row in df_filtered.iterrows():
        if row['brand_name'] == query_brand:
            similarities.append(1.0)#se il prodotto appartiene allo stesso brand avrà similarità massima
        else:
            pair = (query_segment, row[segment_col]) #altrimenti considera il segmento di mercato
            similarities.append(SEGMENT_SIMILARITY.get(pair, 0.0))
    df_filtered['similarity'] = similarities
    return np.array(similarities), df_filtered




4. Applicazione della funzione e creazione dell'aray one-time a partire dal prodotto


In [ ]:
query_id = input('Inserisci il codice del prodotto: ')

brand_sim = brand_segment_similarity_1xK(query_id, df)
#non ottengo una matrice di similarità NXN, ma un array 1xK:
#cioè la similarità del prodotto query rispetto a tutti gli altri prodotti della stessa categoria



#brand_sim è una tupla: il primo elemento è l'array 1xK, il secondo elemento è il dataframe con i prodotti candidati
#brand_sim == (similarity_array, df_filtered).
#la tupla serve a sapere a quali prodotti mi riferisco
brand_sim_array, df_brand_candidates = brand_sim



Inserisci il codice del prodotto: 3245678612507


In [ ]:
brand_sim_array[:10] #vedo i primi dieci valori di similarità

array([0.5, 0.5, 0.5, 0.7, 0.7, 0.7, 0.7, 0.5, 0.7, 0.5])

In [ ]:
df_brand_candidates = df_brand_candidates[[
    'code',
    'query_product_name',
    'query_brand_name',
    'product_name',
    'brand_name',
    'brand_segment',
    'macro_category',
    'similarity'
]]


In [ ]:
df_brand_candidates.sort_values('similarity', ascending= False).head(10) #vedo i primi dieci candidati


,code,query_product_name,query_brand_name,product_name,brand_name,brand_segment,macro_category,similarity
7237,3245678612491,deodorant power 24h,cosmia,deodorant energy 24h,cosmia,middle,Deodorants,1.0
7235,3596710509287,deodorant power 24h,cosmia,deodorant,cosmia,middle,Deodorants,1.0
7247,3245678612682,deodorant power 24h,cosmia,deodorant voluptuous time,cosmia,middle,Deodorants,1.0
7254,3245678612415,deodorant power 24h,cosmia,deodorant instant voluptueux,cosmia,middle,Deodorants,1.0
7260,3245678612743,deodorant power 24h,cosmia,deodorant fleurs blanches 24h,cosmia,middle,Deodorants,1.0
7261,3245678612729,deodorant power 24h,cosmia,deodorant vanille,cosmia,middle,Deodorants,1.0
7232,3245678612439,deodorant power 24h,cosmia,deodorant vanille,cosmia,middle,Deodorants,1.0
7231,3245678612422,deodorant power 24h,cosmia,deodorant grenade,cosmia,middle,Deodorants,1.0
7090,3245678104026,deodorant power 24h,cosmia,deodorant,cosmia,middle,Deodorants,1.0
7057,3245677725475,deodorant power 24h,cosmia,deodorant compresse,cosmia,middle,Deodorants,1.0


5. Con questa funzione si trovano i prodotti più simili a un prodotto dato, guardando solo gli ingredienti. Prima limita il confronto ai prodotti della stessa categoria, così il confronto ha senso.
Poi confronta gli ingredienti del prodotto scelto con quelli degli altri e dice quanto si assomigliano, usando un punteggio numerico.

In [ ]:
#pulizia per esclusione ingredienti che potrebbero compromettere il corretto
#calcolo della similarità
EXCLUDE_INGREDIENTS = {
    'aqua', 'water', 'parfum', 'fragrance',
    'ci', 'color', 'colour',
    'sodium chloride'
}

def clean_ingredients(ing_string):
    ing = [
        i.strip().lower()
        for i in ing_string.split(',')
        if len(i.strip()) > 2
    ]
    ing = [
        i for i in ing
        if not any(x in i for x in EXCLUDE_INGREDIENTS)
        and not i.startswith('ci ')
    ]
    return list(set(ing))

In [ ]:
df_finale['ingredients_list'] = df_finale['ingredients'].apply(clean_ingredients)

df_finale[['ingredients', 'ingredients_list']].head(10)

,ingredients,ingredients_list
0,"Arnica Montana, Avoid Contact With Eyes, Burit...","[arnica montana, avoid contact with eyes, toco..."
1,Oil,[oil]
2,"Added Sugarss, Colorings, Cr 6Ad, Croydon, Lac...","[preservatives, cr 6ad, croydon, lactose, adde..."
3,Oil,[oil]
4,"Lilium Candidum Flower Extract, Oil","[oil, lilium candidum flower extract]"
5,"Allantoin, Aloe Barbadensis Leaf Juice, Benzoi...","[glycerin, phenoxyethanol, hydroxyethyl cellul..."
6,"Coco-Glucoside, Decyl Glucoside, Glycerin, Gly...","[glycerin, glyceryl oleate, sodium cocoyl glut..."
7,Lawsonia Inermis Leaf Powder,[lawsonia inermis leaf powder]
8,Natural Calcium Bentonite Clay,[]
9,"Cellulose Gum, Citric Acid, Cocamidopropyl Bet...","[glycerin, xanthan gum, cellulose gum, tetraso..."


In [ ]:
def jaccard_similarity(set_a, set_b):
    if not set_a or not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)

In [ ]:
def ingredient_similarity_1xK(
    query_product_id,
    df_finale,
    category_col='macro_category'
):
    query_product_id = str(query_product_id)

    if query_product_id not in df_finale['code'].values:
        raise ValueError("Codice prodotto non trovato")

    query_row = df_finale[df_finale['code'] == query_product_id].iloc[0]
    query_category = query_row[category_col]
    query_ing = set(query_row['ingredients_list'])

    # Filtro hard: stessa categoria
    df_filtered = df_finale[df_finale[category_col] == query_category].copy()
    df_filtered = df_filtered[df_filtered['code'] != query_product_id]



    print(f"Trovati {len(df_filtered)} prodotti nella stessa categoria")

    similarities = []

    for _, row in df_filtered.iterrows():
        sim = jaccard_similarity(
            query_ing,
            set(row['ingredients_list'])
        )
        similarities.append(sim)

    return np.array(similarities), df_filtered



In [ ]:
ing_sim_array, df_ing_candidates = ingredient_similarity_1xK(query_id, df_finale)

ing_sim_array[:10]

Trovati 587 prodotti nella stessa categoria


array([0.04      , 0.08333333, 0.        , 0.07692308, 0.08333333,
       0.08333333, 0.        , 0.        , 0.        , 0.11111111])

In [ ]:
#aggiunta similarità e is_query per vedere quali sono i prod raccomandati e
#qual è il prodotto originale cercato
df_ing_candidates_temp = df_ing_candidates[df_ing_candidates['code'] != query_id].copy()
df_ing_candidates_temp['ing_sim'] = ing_sim_array
df_ing_candidates_temp['is_query'] = False
df_ing_candidates_temp = df_ing_candidates_temp.sort_values(by='ing_sim', ascending=False).reset_index(drop=True)


df_query_ing = df_finale[df_finale['code'] == query_id].copy()
df_query_ing['ing_sim'] = 1.0        # similarità massima con se stesso
df_query_ing['is_query'] = True

df_ing_candidates = pd.concat(
    [df_query_ing, df_ing_candidates_temp],
    ignore_index=True
)

df_ing_candidates.head(10)



,code,product_name,macro_category,ingredients,ingredients_list,ing_sim,is_query
0,3245678612507,Déodorant power 24h,Deodorants,"1, 2-Hexanediol, Alcohol Denat, Caprylyl Glyco...","[sodium hydroxide, oil, 2-hexanediol, hydroxye...",1.000000,True
1,3263855884592,Déobille 48h,Deodorants,"Alcohol Denat, Aluminum Chlorohydrate, Citric ...","[sodium hydroxide, aluminum chlorohydrate, oil...",0.444444,False
2,3245678612491,Déodorant Energy 24h,Deodorants,"Alcohol Denat, Allantoin, Citric Acid, Fragran...","[sodium hydroxide, ppg-26-buteth-26, oil, prop...",0.400000,False
3,4005808741601,Déodorant bille,Deodorants,"Alcohol Denat, Aluminum Chlorohydrate, Citric ...","[geraniol, aluminum chlorohydrate, oil, peg-8,...",0.300000,False
4,42349655,Cool Kick deodorant,Deodorants,"Alcohol Denat, Aluminum Chlorohydrate, Citric ...","[geraniol, aluminum chlorohydrate, oil, peg-8,...",0.300000,False
5,42419440,Nivea Deodorant Natural Balance mit bio aloe vera,Deodorants,"Alcohol Denat, Fragrance, Glycerin, Hydroxyeth...","[polyglyceryl-2-caprate, glycerin, hydroxyethy...",0.250000,False
6,4066447365320,Softflower Deodorant,Deodorants,"Alcohol Denat, Benzyl Alcohol, Benzyl Salicyla...","[sodium hydroxide, geraniol, limonene, polygly...",0.250000,False
7,8029041122153,defence deo,Deodorants,"Alcohol Denat, Aluminum Chlorohydrate, Capryly...","[glycerin, lactate, aluminum chlorohydrate, al...",0.222222,False
8,3337871310783,Déodorant Fraîcheur Extrême 24h,Deodorants,"Alcohol Denat, Code Fil C437241, Dipropylene G...","[sodium hydroxide, dipropylene glycol, code fi...",0.222222,False
9,3560070902552,Ocean Deo Stick,Deodorants,"Ci 15985, Ci 42053, Ethylhexylglycerin, Fragra...","[glycerin, sodium hydroxide, tocopherol, oil, ...",0.200000,False


In [ ]:
query_product_ingredients = set(
    df_finale[df_finale['code'] == query_id]['ingredients_list'].iloc[0]
)

def get_common_ingredients(candidate_ingredients):
    return list(query_product_ingredients.intersection(set(candidate_ingredients)))

df_ing_candidates['common_ingr'] = df_ing_candidates['ingredients_list'].apply(get_common_ingredients)

df_ing_candidates[['code', 'product_name', 'ingredients_list', 'common_ingr']].head(10)

,code,product_name,ingredients_list,common_ingr
0,3245678612507,Déodorant power 24h,"[sodium hydroxide, oil, 2-hexanediol, hydroxye...","[sodium hydroxide, oil, 2-hexanediol, hydroxye..."
1,3263855884592,Déobille 48h,"[sodium hydroxide, aluminum chlorohydrate, oil...","[sodium hydroxide, alcohol denat, hydroxyethyl..."
2,3245678612491,Déodorant Energy 24h,"[sodium hydroxide, ppg-26-buteth-26, oil, prop...","[sodium hydroxide, alcohol denat, hydroxyethyl..."
3,4005808741601,Déodorant bille,"[geraniol, aluminum chlorohydrate, oil, peg-8,...","[alcohol denat, hydroxyethylcellulose, oil]"
4,42349655,Cool Kick deodorant,"[geraniol, aluminum chlorohydrate, oil, peg-8,...","[alcohol denat, hydroxyethylcellulose, oil]"
5,42419440,Nivea Deodorant Natural Balance mit bio aloe vera,"[polyglyceryl-2-caprate, glycerin, hydroxyethy...","[hydroxyethylcellulose, alcohol denat]"
6,4066447365320,Softflower Deodorant,"[sodium hydroxide, geraniol, limonene, polygly...","[sodium hydroxide, hydroxyethylcellulose, alco..."
7,8029041122153,defence deo,"[glycerin, lactate, aluminum chlorohydrate, al...","[alcohol denat, caprylyl glycol]"
8,3337871310783,Déodorant Fraîcheur Extrême 24h,"[sodium hydroxide, dipropylene glycol, code fi...","[sodium hydroxide, alcohol denat]"
9,3560070902552,Ocean Deo Stick,"[glycerin, sodium hydroxide, tocopherol, oil, ...","[sodium hydroxide, oil]"


In [ ]:
ing_sim_array[:10]

array([0.04      , 0.08333333, 0.        , 0.07692308, 0.08333333,
       0.08333333, 0.        , 0.        , 0.        , 0.11111111])

In [ ]:
df_ing_candidates.head(10)

,code,product_name,macro_category,ingredients,ingredients_list,ing_sim,is_query,common_ingr
0,3245678612507,Déodorant power 24h,Deodorants,"1, 2-Hexanediol, Alcohol Denat, Caprylyl Glyco...","[sodium hydroxide, oil, 2-hexanediol, hydroxye...",1.000000,True,"[sodium hydroxide, oil, 2-hexanediol, hydroxye..."
1,3263855884592,Déobille 48h,Deodorants,"Alcohol Denat, Aluminum Chlorohydrate, Citric ...","[sodium hydroxide, aluminum chlorohydrate, oil...",0.444444,False,"[sodium hydroxide, alcohol denat, hydroxyethyl..."
2,3245678612491,Déodorant Energy 24h,Deodorants,"Alcohol Denat, Allantoin, Citric Acid, Fragran...","[sodium hydroxide, ppg-26-buteth-26, oil, prop...",0.400000,False,"[sodium hydroxide, alcohol denat, hydroxyethyl..."
3,4005808741601,Déodorant bille,Deodorants,"Alcohol Denat, Aluminum Chlorohydrate, Citric ...","[geraniol, aluminum chlorohydrate, oil, peg-8,...",0.300000,False,"[alcohol denat, hydroxyethylcellulose, oil]"
4,42349655,Cool Kick deodorant,Deodorants,"Alcohol Denat, Aluminum Chlorohydrate, Citric ...","[geraniol, aluminum chlorohydrate, oil, peg-8,...",0.300000,False,"[alcohol denat, hydroxyethylcellulose, oil]"
5,42419440,Nivea Deodorant Natural Balance mit bio aloe vera,Deodorants,"Alcohol Denat, Fragrance, Glycerin, Hydroxyeth...","[polyglyceryl-2-caprate, glycerin, hydroxyethy...",0.250000,False,"[hydroxyethylcellulose, alcohol denat]"
6,4066447365320,Softflower Deodorant,Deodorants,"Alcohol Denat, Benzyl Alcohol, Benzyl Salicyla...","[sodium hydroxide, geraniol, limonene, polygly...",0.250000,False,"[sodium hydroxide, hydroxyethylcellulose, alco..."
7,8029041122153,defence deo,Deodorants,"Alcohol Denat, Aluminum Chlorohydrate, Capryly...","[glycerin, lactate, aluminum chlorohydrate, al...",0.222222,False,"[alcohol denat, caprylyl glycol]"
8,3337871310783,Déodorant Fraîcheur Extrême 24h,Deodorants,"Alcohol Denat, Code Fil C437241, Dipropylene G...","[sodium hydroxide, dipropylene glycol, code fi...",0.222222,False,"[sodium hydroxide, alcohol denat]"
9,3560070902552,Ocean Deo Stick,Deodorants,"Ci 15985, Ci 42053, Ethylhexylglycerin, Fragra...","[glycerin, sodium hydroxide, tocopherol, oil, ...",0.200000,False,"[sodium hydroxide, oil]"


**5. Indicizzazione su codice prodotto**

Costruzione nuovo DF: unione dei due dataframe sul codice prodotto al fine del calcolo della similarità finale.

In [ ]:
df_ing= df_ing_candidates[['code', 'ing_sim']].copy()

df_brand = df_brand_candidates[['code']].copy()
df_brand['brand_sim'] = brand_sim_array

#allineo la similarità
df_merged = df_ing.merge(
    df_brand,
    on='code',
    how='inner'
)


In [ ]:
#check per verificare che l'allineamento è corretto
assert len(df_merged) > 0
assert df_merged['code'].is_unique

df_merged.head()


,code,ing_sim,brand_sim
0,3263855884592,0.444444,0.5
1,3245678612491,0.400000,1.0
2,4005808741601,0.300000,0.5
3,42349655,0.300000,0.5
4,42419440,0.250000,0.5


**6. Calcolo della similarità finale**

Gli ingredienti hanno peso maggiore nel determinare il prodotto più simile (0.8). I brand hanno peso minore (0.2).


In [ ]:
df_merged['final_similarity'] = (
    0.8 * df_merged['ing_sim'] +
    0.2 * df_merged['brand_sim']
)


**6. Costruzione dataframe per la visualizzazione del prodotto più simile**


In [ ]:
query_product_name = (
    df.loc[df['code'] == query_id, 'product_name']
    .iloc[0]
)
risultato = pd.DataFrame({
    'query_product_id': query_id,
    'query_product_name': query_product_name,
    'recommended_product_id': df_merged['code'],
    'final_similarity_score': df_merged['final_similarity'],
    'common_ingredients': df_ing_candidates['common_ingr'],
    'ingredients_query': df_ing_candidates['ingredients']

})

#ordino i prodotti per suggerire il prodotto più simile
risultato = risultato.sort_values(
    by='final_similarity_score',
    ascending=False
).reset_index(drop=True)

risultato['rank'] = risultato.index + 1

# arricchimento informativo (nome, brand, segmento)
risultato = risultato.merge(
    df[['code', 'product_name', 'brand_name', 'brand_segment']],
    left_on='recommended_product_id',
    right_on='code',
    how='left'
).drop(columns='code')

risultato = risultato[
    [
        'rank',
        'query_product_id',
        'query_product_name',
        'ingredients_query',
        'recommended_product_id',
        'product_name',
        'brand_name',
        'brand_segment',
        'common_ingredients',
        'final_similarity_score'
    ]
]

risultato.head(10)



,rank,query_product_id,query_product_name,ingredients_query,recommended_product_id,product_name,brand_name,brand_segment,common_ingredients,final_similarity_score
0,1,3245678612507,deodorant power 24h,"Alcohol Denat, Aluminum Chlorohydrate, Citric ...",3245678612491,deodorant energy 24h,cosmia,middle,"[sodium hydroxide, alcohol denat, hydroxyethyl...",0.520000
1,2,3245678612507,deodorant power 24h,"1, 2-Hexanediol, Alcohol Denat, Caprylyl Glyco...",3263855884592,deobille 48h,sooa,luxury,"[sodium hydroxide, oil, 2-hexanediol, hydroxye...",0.455556
2,3,3245678612507,deodorant power 24h,"Alcohol Denat, Allantoin, Citric Acid, Fragran...",4005808741601,deodorant bille,nivea men,mass_market,"[sodium hydroxide, alcohol denat, hydroxyethyl...",0.340000
3,4,3245678612507,deodorant power 24h,"Alcohol Denat, Aluminum Chlorohydrate, Citric ...",42349655,cool kick deodorant,nivea men,mass_market,"[alcohol denat, hydroxyethylcellulose, oil]",0.340000
4,5,3245678612507,deodorant power 24h,"Alcohol Denat, Citric Acid, Disodium Phosphate...",3245678612439,deodorant vanille,cosmia,middle,"[alcohol denat, hydroxyethylcellulose, oil]",0.333333
5,6,3245678612507,deodorant power 24h,"Arachidic Acid, Butyloctanoic Acid, Distarch P...",3245677725437,deodorant,cosmia,middle,"[sodium hydroxide, oil]",0.333333
6,7,3245678612507,deodorant power 24h,"Alcohol Denat, Aluminum Chlorohydrate, Citric ...",42419440,nivea deodorant natural balance mit bio aloe vera,nivea,mass_market,"[alcohol denat, hydroxyethylcellulose, oil]",0.300000
7,8,3245678612507,deodorant power 24h,"Alcohol Denat, Fragrance, Glycerin, Hydroxyeth...",4066447365320,softflower deodorant,dm balea,mass_market,"[hydroxyethylcellulose, alcohol denat]",0.300000
8,9,3245678612507,deodorant power 24h,"Arachidic Acid, Bht, Caprylyl Glycol, Citric A...",3245678612422,deodorant grenade,cosmia,middle,"[oil, caprylyl glycol]",0.294118
9,10,3245678612507,deodorant power 24h,"2-Methyl 5-Cyclohexylpentanol, Acetic Acid, Ca...",3245677725475,deodorant compresse,cosmia,middle,"[oil, caprylyl glycol]",0.280000
